In [ ]:
# ============================================================
# Notebook 04 - Phase 2: CNN on Log-Mel Spectrograms
# CS-419 Deep Learning Project - Speech Emotion Recognition
# ============================================================
# Experiments:
#   A. CNN without batch normalization
#   B. CNN with batch normalization
#   C. CNN with different filter sizes
#   D. Effect of augmentation (SpecAugment)
# ============================================================

import sys
import os
import numpy as np
import matplotlib.pyplot as plt

sys.path.append("../src")

from models import build_cnn
from train import train_model, plot_history, compare_histories
from evaluate import evaluate_model, plot_model_comparison_bar
from data_loader import get_class_weights, load_tess_paths, split_dataset
from utils import set_seeds, save_results_csv
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, regularizers

set_seeds(42)
os.makedirs("results/plots", exist_ok=True)
os.makedirs("results/models", exist_ok=True)

# ---- Cell 1: Load spectrogram features ----

print("Loading cached spectrograms...")
train_data = np.load("results/cache/train_spec.npz")
val_data   = np.load("results/cache/val_spec.npz")
test_data  = np.load("results/cache/test_spec.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_val,   y_val   = val_data["X"],   val_data["y"]
X_test,  y_test  = test_data["X"],  test_data["y"]

print(f"X_train: {X_train.shape}")
print(f"X_val  : {X_val.shape}")
print(f"X_test : {X_test.shape}")

INPUT_SHAPE = X_train.shape[1:]   # (128, 128, 1)

# ---- Cell 2: Class weights ----

DATA_ROOT = "data/TESS Toronto emotional speech set data"
df = load_tess_paths(DATA_ROOT)
df_train, _, _ = split_dataset(df)
class_weights = get_class_weights(df_train)

# ---- Cell 3: Experiment A - CNN without batch normalization ----

print("\n=== Experiment A: CNN without Batch Normalization ===")

def build_cnn_no_bn(input_shape):
    inp = keras.Input(shape=input_shape)
    x = inp
    for filters in [32, 64, 128, 256]:
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.Conv2D(filters, 3, padding="same", activation="relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(7, activation="softmax")(x)
    model = keras.Model(inp, out, name="CNN_no_BN")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model_a = build_cnn_no_bn(INPUT_SHAPE)
model_a.summary()
hist_a = train_model(model_a, X_train, y_train, X_val, y_val,
                     model_name="CNN_no_BN", epochs=60, batch_size=32,
                     class_weights=class_weights)
res_a = evaluate_model(model_a, X_test, y_test, model_name="CNN_no_BN")

# ---- Cell 4: Experiment B - CNN with Batch Normalization ----

print("\n=== Experiment B: CNN with Batch Normalization ===")

model_b = build_cnn(
    input_shape=INPUT_SHAPE,
    base_filters=32,
    dropout_rate=0.4,
    l2_reg=1e-4,
    optimizer="adam",
    learning_rate=1e-3,
)
model_b.summary()
hist_b = train_model(model_b, X_train, y_train, X_val, y_val,
                     model_name="CNN_with_BN", epochs=60, batch_size=32,
                     class_weights=class_weights)
res_b = evaluate_model(model_b, X_test, y_test, model_name="CNN_with_BN")

# Compare A vs B
compare_histories({"CNN-no-BN": hist_a, "CNN-BN": hist_b}, metric="val_accuracy")

# ---- Cell 5: Experiment C - Depth comparison ----

print("\n=== Experiment C: Shallow (2 blocks) vs Deep (4 blocks) CNN ===")

def build_shallow_cnn(input_shape):
    inp = keras.Input(shape=input_shape)
    x = inp
    for filters in [32, 64]:
        x = layers.Conv2D(filters, 3, padding="same")(x)
        x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D(2)(x)
        x = layers.Dropout(0.2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.4)(x)
    out = layers.Dense(7, activation="softmax")(x)
    model = keras.Model(inp, out, name="CNN_Shallow")
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy",
                  metrics=["accuracy"])
    return model

model_shallow = build_shallow_cnn(INPUT_SHAPE)
hist_shallow = train_model(model_shallow, X_train, y_train, X_val, y_val,
                            model_name="CNN_Shallow", epochs=60, batch_size=32,
                            class_weights=class_weights)
res_shallow = evaluate_model(model_shallow, X_test, y_test, model_name="CNN_Shallow")

compare_histories(
    {"CNN-Shallow (2 blocks)": hist_shallow, "CNN-Deep (4 blocks)": hist_b},
    metric="val_accuracy"
)

# ---- Cell 6: Experiment D - Training with SpecAugment ----

print("\n=== Experiment D: CNN + SpecAugment during training ===")
from augmentation import frequency_mask, time_mask

def augment_batch(X_batch, p=0.5):
    X_aug = X_batch.copy()
    for i in range(len(X_aug)):
        if np.random.rand() < p:
            X_aug[i] = frequency_mask(X_aug[i], max_width=12)
        if np.random.rand() < p:
            X_aug[i] = time_mask(X_aug[i], max_width=15)
    return X_aug

# Create augmented training set (2x the original via offline augmentation)
print("Creating offline augmented training set...")
X_train_aug = augment_batch(X_train, p=0.6)
X_train_combined = np.concatenate([X_train, X_train_aug], axis=0)
y_train_combined = np.concatenate([y_train, y_train], axis=0)

# Shuffle
shuffle_idx = np.random.permutation(len(X_train_combined))
X_train_combined = X_train_combined[shuffle_idx]
y_train_combined = y_train_combined[shuffle_idx]
print(f"Augmented training set: {X_train_combined.shape}")

model_aug = build_cnn(INPUT_SHAPE, base_filters=32, dropout_rate=0.4, l2_reg=1e-4)
hist_aug = train_model(model_aug, X_train_combined, y_train_combined, X_val, y_val,
                       model_name="CNN_augmented", epochs=60, batch_size=32,
                       class_weights=class_weights)
res_aug = evaluate_model(model_aug, X_test, y_test, model_name="CNN_augmented")

# ---- Cell 7: Summary and comparison ----

model_b.save("results/models/cnn_best.keras")
print("Best CNN saved.")

summary = [
    {"model": "CNN (no BN)",    "accuracy": res_a["accuracy"],      "macro_f1": res_a["macro_f1"]},
    {"model": "CNN (with BN)",  "accuracy": res_b["accuracy"],      "macro_f1": res_b["macro_f1"]},
    {"model": "CNN (shallow)",  "accuracy": res_shallow["accuracy"],"macro_f1": res_shallow["macro_f1"]},
    {"model": "CNN (augmented)","accuracy": res_aug["accuracy"],    "macro_f1": res_aug["macro_f1"]},
]
df_results = save_results_csv(summary, "results/cnn_experiment_results.csv")
print(df_results.to_string(index=False))

plot_model_comparison_bar(summary)

print("\n=== Phase 2 Complete ===")
print(f"Best CNN Accuracy : {res_b['accuracy']*100:.2f}%")